In [3]:
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize

load_dotenv(find_dotenv())
client = OpenAI()

In [ ]:
# 1) 인덱스 만들기
def build_faiss_from_df_embeddings(df: pd.DataFrame, emb_col="embedding"):
    vectors = np.vstack(df[emb_col].to_numpy()).astype("float32")

    # 정규화
    vectors = normalize(vectors)

    # 차원 가져오기
    d = vectors.shape[1]
    index = faiss.IndexFlatIP(d)  # 명시적 ID를 저장하지 않음
    index.add(vectors)

    id_map = df.index.to_numpy()
    return index, id_map


# 2) 검색
def search_faiss_with_existing_embeddings(
    df: pd.DataFrame,
    index,
    id_map,
    query: str,
    top_k: int = 3,
    model: str = "text-embedding-3-small",
):
    # 사용자 입력 => embedding
    embedding = client.embeddings.create(input=query).data[0].embedding
    q = np.array(embedding, dtype="float32").reshape(1, -1)

    # 질문 정규화
    q = normalize(q)

    scores, idxs = index.search(q, top_k)
    rows = id_map[idxs[0]]

    # 실제 리뷰 찾기
    result = df.loc[rows].copy()
    result["similarity"] = scores[0]

    return result.sort_values("similarity", ascending=False)


# == 사용 예 ==
index, id_map = build_faiss_from_df_embeddings(amazon_df, emb_col="embedding")

res = search_faiss_with_existing_embeddings(
    amazon_df, index, id_map, query="맛있는 콩", top_k=3
)

res

### 네이버 영화 리뷰

In [ ]:
# 1000개의 샘플링 데이터 가져오기
train_sample = pd.read_csv("./data/train_sample.csv")

# 임베딩 데이터 가져오기
X_train = np.load("./data/X_train.npy")

In [ ]:
# --- cosine을 위해 L2 정규화
vectors = normalize(X_train)

# 인덱스 만들기
d = vectors.shape[1]
index = faiss.IndexFlatIP(d)
index.add(vectors)

# 사용자 입력
query = "그냥저냥;;"
model = "text-embedding-3-small"
embedding = client.embeddings.create(input=query, model=model).data[0].embedding
q = np.array(embedding, dtype="float32").reshape(1, -1)

# 질문 정규화
q = normalize(q)

scores, idxs = index.search(q, 5)

rows = idxs[0]
scores = scores[0]

for idx, no in enumerate(rows):
    doc = train_sample.iloc[no]["document"]
    label = train_sample.iloc[no]["label"]
    sim = scores[idx]
    print(f"sim={sim}, label={label}, doc={doc}")